# 01 — Dataset summary

Produces **Table 1** (`tab:sample`) of the paper: per-band, per-epoch counts of
nights, visits and raft-stacked PSF stamps, with date ranges.

Also reports the distribution of `n_stack` (stars averaged per raft stamp), quoted
in Section `data:stamps`.

Reads only `../../data/shapelet_bvec/shapelet_{band}_{tag}.npz` and a random sample
of `../../data/stamps_stack_51/stamps_*.npz`. Nothing is re-fitted here.

In [1]:
import importlib.util, sys
spec = importlib.util.spec_from_file_location('plot_config', 'plot_config.py')
plot_config = importlib.util.module_from_spec(spec); spec.loader.exec_module(plot_config)
from plot_config import BANDS, DATA, bvec_path, STAMPS_DIR, apply_style

import numpy as np
import pandas as pd

apply_style()

EPOCHS = {'DP2': 'dp2all', 'Post-DP2': 'postdp2'}
print('data dir:', DATA)

data dir: /sdf/data/rubin/user/ztq1996/psf-rubin/psf_zernike/data


## Per-band, per-epoch counts

In [2]:
def fmt_date(d):
    s = str(int(d))
    return f'{s[:4]}-{s[4:6]}-{s[6:]}'

rows = []
for epoch, tag in EPOCHS.items():
    for band in BANDS:
        d = np.load(bvec_path(band, tag))
        v = d['visit']
        nights = np.unique(v // 100_000)
        rows.append(dict(epoch=epoch, band=band,
                         nights=len(nights),
                         visits=len(np.unique(v)),
                         psfs=len(v),
                         first=fmt_date(nights.min()),
                         last=fmt_date(nights.max())))

tab = pd.DataFrame(rows)

# Totals per epoch (visits and nights de-duplicated across bands)
for epoch, tag in EPOCHS.items():
    vs, ns = set(), set()
    for band in BANDS:
        v = np.load(bvec_path(band, tag))['visit']
        vs |= set(v.tolist()); ns |= set((v // 100_000).tolist())
    sub = tab[tab.epoch == epoch]
    print(f'{epoch:>9}: nights={len(ns):>4}  visits={len(vs):>6}  PSFs={sub.psfs.sum():>8,}')

print(f'\n    TOTAL: visits={tab.visits.sum():,}  PSFs={tab.psfs.sum():,}')
tab

      DP2: nights= 130  visits= 29502  PSFs= 612,511


 Post-DP2: nights= 122  visits= 64920  PSFs=1,336,678

    TOTAL: visits=94,422  PSFs=1,949,189


,epoch,band,nights,visits,psfs,first,last
0,DP2,u,38,2056,42698,2025-04-24,2025-12-20
1,DP2,g,79,4280,88863,2025-04-24,2026-01-06
2,DP2,r,76,5063,104521,2025-04-24,2026-01-05
3,DP2,i,99,7899,164443,2025-04-24,2026-01-06
4,DP2,z,79,5867,121771,2025-05-15,2026-01-05
5,DP2,y,29,4337,90215,2025-05-19,2026-01-06
6,Post-DP2,u,32,2511,51496,2026-01-13,2026-07-13
7,Post-DP2,g,71,4273,88708,2026-01-02,2026-07-13
8,Post-DP2,r,83,9273,191248,2026-01-03,2026-07-13
9,Post-DP2,i,109,24536,506259,2026-01-01,2026-07-13


## Stars per raft stamp

Each stamp is the flux-normalised mean of up to 100 stars on one detector of one raft.
Sample the stamp files to characterise the actual stacking depth.

In [3]:
# NOTE: the stamp directory holds ~76k files on a shared filesystem; both the
# directory listing and each read cost ~0.5-1 s. Keep N_SAMPLE small -- 150
# files is ~3000 raft stamps, far more than enough for these summary statistics.
N_SAMPLE = 150

rng = np.random.default_rng(42)
files = sorted(STAMPS_DIR.glob('stamps_*.npz'))
sample = rng.choice(len(files), size=min(N_SAMPLE, len(files)), replace=False)

nstack, nraft, shp = [], [], set()
for i in sample:
    d = np.load(files[i], allow_pickle=True)
    nstack.append(d['n_stack']); nraft.append(len(d['n_stack']))
    shp.add(d['stamps'].shape[1:])
nstack = np.concatenate(nstack)

print(f'stamp files on disk       : {len(files):,}')
print(f'stamp shape               : {shp}')
print(f'files sampled             : {len(sample)}  ({len(nstack):,} raft stamps)')
print(f'rafts per visit (median)  : {np.median(nraft):.0f}  (range {min(nraft)}-{max(nraft)})')
print(f'stars per stamp   median  : {np.median(nstack):.0f}')
print(f'                  16-84%  : {np.percentile(nstack,16):.0f} - {np.percentile(nstack,84):.0f}')
print(f'                  range   : {nstack.min()} - {nstack.max()}')

stamp files on disk       : 99,336
stamp shape               : {(51, 51)}
files sampled             : 150  (3,136 raft stamps)
rafts per visit (median)  : 21  (range 17-21)
stars per stamp   median  : 49
                  16-84%  : 26 - 100
                  range   : 1 - 100


### Stacking depth per epoch

The pooled median above mixes the two epochs, whose PSF-star catalogs differ: DP2 uses
`refit_psf_star`, post-DP2 uses `single_visit_star` with `calib_psf_used`. Sample each
epoch's own visits separately so the stacking depth can be quoted per epoch.

In [4]:
N_PER_EPOCH = 100   # visit files per epoch (each ~21 raft stamps)
rng = np.random.default_rng(42)
for epoch, tag in EPOCHS.items():
    v = np.unique(np.concatenate([np.load(bvec_path(b, tag))['visit'] for b in BANDS]))
    ns = []
    for vid in rng.choice(v, N_PER_EPOCH, replace=False):
        ns.append(np.load(STAMPS_DIR / f'stamps_{vid}.npz', allow_pickle=True)['n_stack'])
    ns = np.concatenate(ns)
    print(f'{epoch:>9}: stars per stamp median = {np.median(ns):.0f}   '
          f'16-84% = {np.percentile(ns, 16):.0f}-{np.percentile(ns, 84):.0f}   '
          f'at the 100-star cap = {(ns == 100).mean() * 100:.0f}%   ({len(ns):,} raft stamps)')


      DP2: stars per stamp median = 100   16-84% = 80-100   at the 100-star cap = 53%   (2,081 raft stamps)


 Post-DP2: stars per stamp median = 43   16-84% = 22-69   at the 100-star cap = 3%   (2,079 raft stamps)


## LaTeX table for the paper

In [5]:
# LaTeX rows for Table 1. Nights and visits in the 'all' row are de-duplicated
# across bands, so they are recomputed here rather than summed down the column.
# First/last night columns were dropped from the paper table (2026-11-06).
lines = []
for epoch, tag in EPOCHS.items():
    sub = tab[tab.epoch == epoch]
    lines.append(r'\multicolumn{4}{l}{\textit{' + epoch + r'}} \\')
    for _, r in sub.iterrows():
        lines.append(f"\\quad ${r['band']}$ & {r['nights']} & {r['visits']:,} "
                     f"& {r['psfs']:,} " + r'\\')
    vs, ns = set(), set()
    for band in BANDS:
        v = np.load(bvec_path(band, tag))['visit']
        vs |= set(v.tolist()); ns |= set((v // 100_000).tolist())
    lines.append(f"\\quad all & {len(ns)} & {len(vs):,} & {sub['psfs'].sum():,} " + r'\\')
    lines.append(r'\addlinespace')
print('\n'.join(lines))

\multicolumn{4}{l}{\textit{DP2}} \\
\quad $u$ & 38 & 2,056 & 42,698 \\
\quad $g$ & 79 & 4,280 & 88,863 \\
\quad $r$ & 76 & 5,063 & 104,521 \\
\quad $i$ & 99 & 7,899 & 164,443 \\
\quad $z$ & 79 & 5,867 & 121,771 \\
\quad $y$ & 29 & 4,337 & 90,215 \\
\quad all & 130 & 29,502 & 612,511 \\
\addlinespace
\multicolumn{4}{l}{\textit{Post-DP2}} \\
\quad $u$ & 32 & 2,511 & 51,496 \\
\quad $g$ & 71 & 4,273 & 88,708 \\
\quad $r$ & 83 & 9,273 & 191,248 \\
\quad $i$ & 109 & 24,536 & 506,259 \\
\quad $z$ & 91 & 14,509 & 295,478 \\
\quad $y$ & 44 & 9,818 & 203,489 \\
\quad all & 122 & 64,920 & 1,336,678 \\
\addlinespace
